# RAG Pipeline Evaluation Project

# ETL part

load EnterpriseRAG-Bench from Hugging Face

In [8]:
# Step 1: Install required library
# "datasets" is the Hugging Face library used to download public datasets.
!pip install datasets -q

## 1. Load EnterpriseRAG-Bench Dataset
This section installs required libraries and loads the documents/questions datasets from Hugging Face.

In [9]:
# Step 2: Import required libraries

# load_dataset is used to load datasets from Hugging Face.
from datasets import load_dataset

# pandas is used to convert the dataset into table format.
import pandas as pd


# Step 3: Load EnterpriseRAG-Bench dataset from Hugging Face

# This dataset has two important parts:
# 1. documents  -> company-like documents used as RAG knowledge base
# 2. questions  -> test questions with expected answers and expected document IDs

documents_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "documents",
    split="test"
)

questions_dataset = load_dataset(
    "onyx-dot-app/EnterpriseRAG-Bench",
    "questions",
    split="test"
)


# Step 4: Convert Hugging Face dataset into pandas DataFrame
# DataFrame means table format, like Excel/CSV.

documents_df = documents_dataset.to_pandas()
questions_df = questions_dataset.to_pandas()


# Step 5: Check how many rows and columns are present

print("Documents dataset shape:", documents_df.shape)
print("Questions dataset shape:", questions_df.shape)


# Step 6: Show column names
# This helps us understand what fields are available.

print("\nDocuments columns:")
print(documents_df.columns.tolist())

print("\nQuestions columns:")
print(questions_df.columns.tolist())


# Step 7: Preview first few rows

print("\nDocuments preview:")
display(documents_df.head())

print("\nQuestions preview:")
display(questions_df.head())

Documents dataset shape: (511962, 4)
Questions dataset shape: (500, 7)

Documents columns:
['doc_id', 'source_type', 'title', 'content']

Questions columns:
['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Documents preview:


,doc_id,source_type,title,content
0,dsid_e54ef48bae78474684a957cf613d47d5,confluence,Runbook: Deploy / Upgrade / Roll Back perf-can...,## Purpose\nThis runbook describes the operati...
1,dsid_229dd48e9b1d466a81ebaffe3ec84469,confluence,Cross-account GPU burst SLO contract and CI li...,Summary\n\nOverview:\nThis document defines th...
2,dsid_aeb0022d62bc43beb6549ba92e5655eb,confluence,Third-Party & Vendor Coordination Playbook for...,Overview\n\nPurpose:\nThis playbook documents ...
3,dsid_926174fc4900408c89c98abde46b7225,confluence,First Production Launch Checklist (Canonical T...,## Purpose\nThis page defines the **canonical ...
4,dsid_d511c6d8daf94f998ce6a3d97462af2d,confluence,Go-Live Runway and Week-3 Stability Playbook,Overview\n\nThis playbook defines the technica...



Questions preview:


,question_id,question_type,source_types,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,[github],What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,[github],What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,[linear],What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,[fireflies],In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,[gmail],What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...


## 2. Inspect Dataset Structure
This section checks one sample document and one sample question to understand the dataset fields.

In [10]:
# Step 8: Check one full document row
# This helps us understand what one knowledge-base document looks like.

print("One document example:")
display(documents_df.iloc[0])


# Step 9: Check one full question row
# This helps us understand what one test question looks like.

print("One question example:")
display(questions_df.iloc[0])

One document example:


,0
doc_id,dsid_e54ef48bae78474684a957cf613d47d5
source_type,confluence
title,Runbook: Deploy / Upgrade / Roll Back perf-can...
content,## Purpose\nThis runbook describes the operati...


One question example:


,0
question_id,qst_0001
question_type,basic
source_types,[github]
question,What are the default size limits for file uplo...
expected_doc_ids,[dsid_ae068ee4aa9640159427cd941bef0238]
gold_answer,The default limits are 10 MiB per file (max_fi...
answer_facts,[The default per file upload size limit (max_f...


## 3. Select Lightweight QA Test Questions
This section selects 10 basic questions with one clear expected source document.

In [11]:
# Step 10: Select simple/basic questions for our first QA test set

# We are starting with "basic" questions because they are easier to verify.
# Later, we can test complex question types.

basic_questions_df = questions_df[questions_df["question_type"] == "basic"].copy()


# Step 11: Keep only questions that have exactly one expected document
# This makes the first version simple:
# one question -> one correct source document -> one expected answer.

basic_questions_df["expected_doc_count"] = basic_questions_df["expected_doc_ids"].apply(len)

single_doc_questions_df = basic_questions_df[
    basic_questions_df["expected_doc_count"] == 1
].copy()


# Step 12: Select first 10 questions
# We are choosing only 10 because this is the first QA prototype.
# After this works, we can increase the count.

selected_questions_df = single_doc_questions_df.head(10).copy()


# Step 13: Show selected questions

print("Selected questions count:", len(selected_questions_df))

display(
    selected_questions_df[
        [
            "question_id",
            "question_type",
            "question",
            "expected_doc_ids",
            "gold_answer",
            "answer_facts"
        ]
    ]
)

Selected questions count: 10


,question_id,question_type,question,expected_doc_ids,gold_answer,answer_facts
0,qst_0001,basic,What are the default size limits for file uplo...,[dsid_ae068ee4aa9640159427cd941bef0238],The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...
1,qst_0002,basic,What is the name of the new metric added so SR...,[dsid_9550250a59e74f1bbd5612480b2e7100],The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...
2,qst_0003,basic,What are the acceptance criteria for the proje...,[dsid_3fd6af404fae48e6b8ea5a57875ef78f],The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...
3,qst_0004,basic,In the meeting about onboarding a SaaS product...,[dsid_6c4c1c875e704f09b4d791d64d7bc7e5],The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...
4,qst_0005,basic,What failover sequence and recovery targets di...,[dsid_8e838ab6a98f4cbcb672d41f210ff89c],MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...
5,qst_0006,basic,In the draft spec about extending a routing po...,[dsid_184be937d34a412ab5e61366d54d8ed6],The draft proposes this canonical v1 signal pr...,[The draft spec proposes a canonical v1 priori...
6,qst_0007,basic,In a rolling investigation of a model regressi...,[dsid_72ec4a9962ba43e88acd61abbba1052d],"In the first comparison run, the baseline buil...","[In the first comparison run, the older baseli..."
7,qst_0008,basic,"In the internal shiproom runner notes, what is...",[dsid_5fc2dba9f6ac4af2b49b4f546a4298d0],The rollback plan is considered verified in st...,"[In staging, a verified rollback requires a ti..."
8,qst_0009,basic,"In the EdgePath evaluation email thread, what ...",[dsid_85deb10a652742baaf28af6149600001],Redwood said it wouldn't match the competitor'...,[Redwood did not match the competitors 50 perc...
9,qst_0010,basic,How does the new alerting approach group model...,[dsid_c1a6a71323c04c1ba5445aadea340362],It adds a lightweight token_stage_cohort servi...,[The approach adds a lightweight token_stage_c...


## 4. Create Lightweight QA Dataset
This section maps each selected question to its expected source document and prepares the QA dataset.

In [12]:
# Step 14: Get the matching source document for each selected question

# We create a lookup table using doc_id.
# This helps us quickly find a document by its document ID.

documents_lookup = documents_df.set_index("doc_id")


# Step 15: Create rows for our lightweight QA dataset

lightweight_rows = []

for index, question_row in selected_questions_df.iterrows():

    # Each selected question has exactly one expected document ID.
    expected_doc_id = question_row["expected_doc_ids"][0]

    # Find the matching document from documents_df using that expected_doc_id.
    matching_document = documents_lookup.loc[expected_doc_id]

    # Create one QA test row.
    lightweight_rows.append({
        "test_id": f"RAG_TC_{len(lightweight_rows) + 1:03d}",

        # Unique code created by us for easy tracking.
        "input_code": f"EVL-RAG-{len(lightweight_rows) + 1:03d}",

        # Question details from questions dataset.
        "question_id": question_row["question_id"],
        "question_type": question_row["question_type"],
        "question": question_row["question"],

        # Expected source document details.
        "expected_doc_ids": expected_doc_id,
        "expected_doc_title": matching_document["title"],
        "expected_source_type": matching_document["source_type"],
        "expected_doc_content": matching_document["content"],

        # Expected answer details.
        "gold_answer": question_row["gold_answer"],
        "answer_facts": question_row["answer_facts"],

        # These fields will be filled after running the RAG chatbot/model.
        "actual_retrieved_doc_ids": "",
        "actual_output": "",

        # These fields will be filled by our embedding evaluator later.
        "expected_embedding_code": "",
        "actual_embedding_code": "",
        "similarity_score": "",

        # Final QA result fields.
        "faithfulness_check": "Not Checked",
        "hallucination_check": "Not Checked",
        "match_status": "Not Run",
        "remarks": ""
    })


# Step 16: Convert rows into DataFrame

lightweight_rag_df = pd.DataFrame(lightweight_rows)


# Step 17: Preview the lightweight QA dataset

print("Lightweight QA dataset shape:", lightweight_rag_df.shape)

display(lightweight_rag_df.head())

Lightweight QA dataset shape: (10, 20)


,test_id,input_code,question_id,question_type,question,expected_doc_ids,expected_doc_title,expected_source_type,expected_doc_content,gold_answer,answer_facts,actual_retrieved_doc_ids,actual_output,expected_embedding_code,actual_embedding_code,similarity_score,faithfulness_check,hallucination_check,match_status,remarks
0,RAG_TC_001,EVL-RAG-001,qst_0001,basic,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",github,description:\nMotivation: users integrating to...,The default limits are 10 MiB per file (max_fi...,[The default per file upload size limit (max_f...,,,,,,Not Checked,Not Checked,Not Run,
1,RAG_TC_002,EVL-RAG-002,qst_0002,basic,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,github,description:\nContext: production customers ob...,The new metric is `stream.timebox_finalized` (...,[The new metric added is named stream.timebox_...,,,,,,Not Checked,Not Checked,Not Run,
2,RAG_TC_003,EVL-RAG-003,qst_0003,basic,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,linear,description:\nObjective: Create a deterministi...,The acceptance criteria are: (1) deliver a sta...,[Deliver a stable design JSON token spec that ...,,,,,,Not Checked,Not Checked,Not Run,
3,RAG_TC_004,EVL-RAG-004,qst_0004,basic,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,fireflies,summary:\nRedwood and the GCP Marketplace team...,The GCP team said entitlement propagation dela...,[Entitlement propagation delays can occur afte...,,,,,,Not Checked,Not Checked,Not Run,
4,RAG_TC_005,EVL-RAG-005,qst_0005,basic,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,gmail,['From: Rafael Mendes <rafael.mendes@redwoodin...,MedThink's preferred failover hierarchy is EU ...,[MedThink specified a failover hierarchy of EU...,,,,,,Not Checked,Not Checked,Not Run,


In [13]:
# Step 18: Save the lightweight QA dataset as a CSV file

# This CSV is our final ETL output.
# We will use this file for QA testing and embedding comparison later.

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset.csv", index=False)


# Step 19: Confirm the file is saved

print("CSV file created successfully: lightweight_rag_qa_dataset.csv")

CSV file created successfully: lightweight_rag_qa_dataset.csv


Embedding evaluator part

## 5. Generate Expected Embedding Fingerprint Codes
This section converts gold answers into expected embedding fingerprint codes.

In [14]:
# Step 20: Install sentence-transformers
# This library gives us an embedding model.
# Embedding means converting text into numeric meaning code/vector.

!pip install sentence-transformers -q

In [15]:
# Step 21: Load embedding model

# SentenceTransformer is used to convert text into embedding vectors.
from sentence_transformers import SentenceTransformer

# This is a small and commonly used embedding model.
# It converts sentence meaning into a numeric vector.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [16]:
# Step 22: Create expected embedding fingerprint code from gold_answer

# hashlib is used to create a small unique fingerprint from the full embedding.
# This makes the code short and readable, while still being based on the full embedding.
import hashlib
import numpy as np


def create_embedding_fingerprint(text, prefix="EXP"):
    # If text is empty or missing, return empty code.
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # Convert the full text meaning into an embedding vector.
    # This vector represents the meaning of the expected answer.
    embedding_vector = embedding_model.encode(str(text))

    # Round the full embedding vector slightly before hashing.
    # This keeps the fingerprint stable and avoids tiny decimal noise.
    rounded_vector = np.round(embedding_vector, 6)

    # Convert the full rounded vector into bytes.
    # Hashing needs byte format, so we convert the numeric vector into bytes.
    vector_bytes = rounded_vector.tobytes()

    # Create a short hash/fingerprint from the full embedding vector.
    # We use only the first 8 characters to keep the code simple.
    fingerprint = hashlib.sha256(vector_bytes).hexdigest()[:8]

    # Return a small readable embedding code.
    # Example: EXP-a91f23c8
    return f"{prefix}-{fingerprint}"


# Apply the embedding fingerprint generator to gold_answer.
# This creates a short expected embedding code based on the full expected answer meaning.
lightweight_rag_df["expected_embedding_code"] = lightweight_rag_df["gold_answer"].apply(
    lambda text: create_embedding_fingerprint(text, prefix="EXP")
)


# Show the generated expected embedding codes.
display(
    lightweight_rag_df[
        ["test_id", "gold_answer", "expected_embedding_code"]
    ].head()
)


,test_id,gold_answer,expected_embedding_code
0,RAG_TC_001,The default limits are 10 MiB per file (max_fi...,EXP-0b583284
1,RAG_TC_002,The new metric is `stream.timebox_finalized` (...,EXP-e07dadfc
2,RAG_TC_003,The acceptance criteria are: (1) deliver a sta...,EXP-f4193af0
3,RAG_TC_004,The GCP team said entitlement propagation dela...,EXP-4fe10deb
4,RAG_TC_005,MedThink's preferred failover hierarchy is EU ...,EXP-77f7105f


In [17]:
# Step 23: Save updated CSV with expected embedding codes

lightweight_rag_df.to_csv("lightweight_rag_qa_dataset_with_expected_codes.csv", index=False)

print("Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv")

Updated CSV saved successfully: lightweight_rag_qa_dataset_with_expected_codes.csv


 Developer/QA note:
 Before running the comparison step, paste or load the RAG model outputs into the actual_output column.
 actual_output should contain the answer returned by the RAG chatbot/model for each question.

## 6. Prepare for Actual Output Comparison
This section loads the prepared CSV and checks whether actual model outputs are available.

In [18]:
from google.colab import files

uploaded = files.upload()

Saving lightweight_rag_qa_dataset_with_expected_codes (7).csv to lightweight_rag_qa_dataset_with_expected_codes (7).csv


In [19]:
import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")

print("CSV loaded successfully")
print(qa_df.shape)

CSV loaded successfully
(10, 20)


In [20]:

# Step 24: Load the updated CSV for actual-output comparison stage

# This CSV already contains:
# - selected RAG test questions
# - expected source documents
# - gold_answer
# - expected_embedding_code
# - empty fields for actual_output and comparison results

import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")


# Step 25: Check current QA dataset status

# Total number of test cases available in the CSV.
total_test_cases = len(qa_df)

# Count rows where actual_output is empty.
# actual_output must be filled only after running the RAG model/chatbot.
empty_actual_output_count = qa_df["actual_output"].isna().sum() + (
    qa_df["actual_output"].astype(str).str.strip() == ""
).sum()

# Count rows where expected_embedding_code is already created.
expected_code_count = (
    qa_df["expected_embedding_code"].notna()
    & (qa_df["expected_embedding_code"].astype(str).str.strip() != "")
).sum()


print("Total test cases:", total_test_cases)
print("Rows with expected embedding code:", expected_code_count)
print("Rows waiting for actual model output:", empty_actual_output_count)


# Step 26: Display important QA tracking columns

# This preview helps us verify that:
# - expected answer code is already available
# - actual_output is still waiting for model response
# - result columns are ready for the next comparison phase

display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "expected_embedding_code",
            "actual_output",
            "actual_embedding_code",
            "similarity_score",
            "match_status"
        ]
    ].head()
)


Total test cases: 10
Rows with expected embedding code: 10
Rows waiting for actual model output: 10


,test_id,question,gold_answer,expected_embedding_code,actual_output,actual_embedding_code,similarity_score,match_status
0,RAG_TC_001,What are the default size limits for file uplo...,The default limits are 10 MiB per file (max_fi...,EXP-0b583284,NaN,NaN,NaN,Not Run
1,RAG_TC_002,What is the name of the new metric added so SR...,The new metric is `stream.timebox_finalized` (...,EXP-e07dadfc,NaN,NaN,NaN,Not Run
2,RAG_TC_003,What are the acceptance criteria for the proje...,The acceptance criteria are: (1) deliver a sta...,EXP-f4193af0,NaN,NaN,NaN,Not Run
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,The GCP team said entitlement propagation dela...,EXP-4fe10deb,NaN,NaN,NaN,Not Run
4,RAG_TC_005,What failover sequence and recovery targets di...,MedThink's preferred failover hierarchy is EU ...,EXP-77f7105f,NaN,NaN,NaN,Not Run


In [21]:
# Convert these columns into text type before storing text values.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")

## Demo RAG Pipeline Implementation Plan

This section creates a simple demo RAG pipeline only for testing the QA evaluation workflow.

Implementation flow:

1. Create a `document_store` folder.
2. Export each expected document from the CSV into a separate `.txt` file inside `document_store`.
3. Add the document file path back into the dataset as `expected_doc_path`.
4. Add correct documents and later add distractor/wrong documents into the same `document_store`.
5. Demo RAG will read documents from the `document_store` folder.
6. Retrieval will use hybrid search:
   - keyword matching score
   - embedding similarity score
7. The best matching document will be selected as `actual_retrieved_doc_ids`.
8. The answer will be created by extracting the most relevant sentences from the selected document.
9. The generated answer will be stored as `actual_output`.
10. Existing QA checks will run:
    - retrieval match
    - semantic similarity
    - fact-level meaning check
    - overall QA status

Note:

This is not a production RAG chatbot. It is a QA test setup created to generate actual outputs and validate the RAG evaluation workflow end-to-end.

In [22]:
# Step 27: Create document_store folder and export expected documents as .txt files

# This step separates document content from the CSV.
# CSV will keep QA test case details.
# document_store folder will keep source documents like a real RAG knowledge base.

import os
import re

# Create document_store folder if it does not already exist
document_store_path = "document_store"
os.makedirs(document_store_path, exist_ok=True)


def clean_filename(text):
    """
    Create a safe file name from document id/title.
    This removes characters that are not safe for file names.
    """
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_-]", "_", text)
    return text[:120]


# Create a new column to store each exported document file path
qa_df["expected_doc_path"] = ""

# Export each expected document content into a separate text file
for index, row in qa_df.iterrows():
    test_id = row["test_id"]
    doc_id = row["expected_doc_ids"]
    doc_title = row["expected_doc_title"]
    doc_content = row["expected_doc_content"]

    # Create readable and unique file name
    file_name = f"{test_id}_{clean_filename(doc_id)}.txt"
    file_path = os.path.join(document_store_path, file_name)

    # Write document content into .txt file
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(f"doc_id: {doc_id}\n")
        file.write(f"title: {doc_title}\n\n")
        file.write(str(doc_content))

    # Store file path back into dataframe
    qa_df.loc[index, "expected_doc_path"] = file_path


print("Document store created successfully.")
print("Total documents exported:", len(qa_df))
print("Folder name:", document_store_path)

display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "expected_doc_title",
            "expected_doc_path"
        ]
    ]
)

Document store created successfully.
Total documents exported: 10
Folder name: document_store


,test_id,expected_doc_ids,expected_doc_title,expected_doc_path
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,"add multipart/form-data handling, strict conte...",document_store/RAG_TC_001_dsid_ae068ee4aa96401...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,introduce-server-timebox-and-idempotent-cancel...,document_store/RAG_TC_002_dsid_9550250a59e74f1...
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,Develop interactive tone derivatives and Kappa...,document_store/RAG_TC_003_dsid_3fd6af404fae48e...
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,GCP Marketplace onboarding + billing review (R...,document_store/RAG_TC_004_dsid_6c4c1c875e704f0...
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,Regional fallback priorities & logs — post-cal...,document_store/RAG_TC_005_dsid_8e838ab6a98f4cb...
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,Draft Spec: Policy Engine Extensions for Regio...,document_store/RAG_TC_006_dsid_184be937d34a412...
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,rolling-bias-bisection-log-jared,document_store/RAG_TC_007_dsid_72ec4a9962ba43e...
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,Shiproom runner: personal prep log and sticky ...,document_store/RAG_TC_008_dsid_5fc2dba9f6ac4af...
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,Licensing offsets & packaging for potential mi...,document_store/RAG_TC_009_dsid_85deb10a652742b...
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,introduce-token-stage-cohorting-and-route-matr...,document_store/RAG_TC_010_dsid_c1a6a71323c04c1...


In [23]:
# Step 28: Check exported documents inside document_store folder

# This confirms that our document files are created properly.
# The demo RAG system will read documents from this folder.

exported_files = os.listdir(document_store_path)

print("Total files in document_store:", len(exported_files))
print("First few files:")

for file_name in exported_files[:5]:
    print(file_name)

Total files in document_store: 10
First few files:
RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
RAG_TC_008_dsid_5fc2dba9f6ac4af2b49b4f546a4298d0.txt
RAG_TC_007_dsid_72ec4a9962ba43e88acd61abbba1052d.txt
RAG_TC_004_dsid_6c4c1c875e704f09b4d791d64d7bc7e5.txt
RAG_TC_005_dsid_8e838ab6a98f4cbcb672d41f210ff89c.txt


In [24]:
# Step 29: Load documents from document_store folder

# This step reads the exported .txt files.
# Now the demo RAG system will use file-based documents instead of reading document content directly from CSV.

document_store = []

for file_name in os.listdir(document_store_path):
    if file_name.endswith(".txt"):
        file_path = os.path.join(document_store_path, file_name)

        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()

        document_store.append(
            {
                "file_name": file_name,
                "file_path": file_path,
                "content": content
            }
        )

print("Documents loaded from document_store:", len(document_store))

# Show one loaded document sample
print("Sample file:", document_store[0]["file_name"])
print("Sample content preview:")
print(document_store[0]["content"][:500])

Documents loaded from document_store: 10
Sample file: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Sample content preview:
doc_id: dsid_ae068ee4aa9640159427cd941bef0238
title: add multipart/form-data handling, strict content-type validation, and payload limit enforcement for API tool/file inputs

description:
Motivation: users integrating tool/function calling and file uploads via the OpenAI-compatibility endpoints were sending a wide variety of content-types and very large multipart requests which caused inconsistent runtime behavior, high memory pressure, and unexpected acceptance of unsupported payloads. This PR 


In [25]:
# Step 29A: Create document embeddings once for vector retrieval

# This follows proper RAG architecture.
# Documents are loaded from document_store folder.
# Then all documents are converted into embeddings one time.
# During retrieval, we will reuse these embeddings.

document_texts = [document["content"] for document in document_store]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_tensor=True
)

print("Document embeddings created successfully.")
print("Total documents embedded:", len(document_texts))
print("Total embedding vectors:", len(document_embeddings))

Document embeddings created successfully.
Total documents embedded: 10
Total embedding vectors: 10


In [26]:
# Step 30: Create vector retrieval logic

# This step retrieves the top-k most relevant documents for a question.
#
# Flow:
# 1. User question is converted into an embedding.
# 2. The question embedding is compared with all stored document embeddings.
# 3. Documents are ranked by similarity score.
# 4. Top-k documents are returned in ranked order.

from sentence_transformers import util


def vector_retrieve(question, document_store, document_embeddings, top_k=3):
    """
    Retrieves top-k relevant documents for the given question.

    Input:
    - question: user question
    - document_store: documents loaded from document_store folder
    - document_embeddings: embeddings created once for all documents
    - top_k: number of documents to retrieve

    Output:
    - best_document: rank 1 document
    - top_k_results: top-k documents in ranked order
    """

    # The user question is normal text.
    # To compare it with documents, we first convert the question into vector format.
    # This vector represents the meaning of the question.
    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    # Now compare the question vector with all document vectors.
    # This gives one similarity score for each document.
    # Higher score means that document is more related to the question.
    similarity_scores = util.cos_sim(
        question_embedding,
        document_embeddings
    )[0]

    retrieval_results = []

    # Store each document with its similarity score.
    # This makes the ranking visible for QA checking.
    # In some vector databases, this ranking happens internally.
    # Here we store it manually so we can inspect rank, file name, and score.
    for index, document in enumerate(document_store):
        retrieval_results.append(
            {
                "rank": index + 1,
                "file_name": document["file_name"],
                "file_path": document["file_path"],
                "content": document["content"],
                "similarity_score": round(float(similarity_scores[index]), 4)
            }
        )

    # Sort documents by highest similarity score.
    # Rank 1 should be the most relevant document.
    retrieval_results = sorted(
        retrieval_results,
        key=lambda item: item["similarity_score"],
        reverse=True
    )

    # Update rank after sorting.
    # After sorting, rank 1 means best matching document.
    for rank_index, item in enumerate(retrieval_results):
        item["rank"] = rank_index + 1

    # Keep only top-k documents.
    # Example: if top_k=3, keep only top 3 retrieved documents.
    top_k_results = retrieval_results[:top_k]

    # First document is the best retrieved document.
    best_document = top_k_results[0]

    return best_document, top_k_results


# Test vector retrieval for first question.
sample_question = qa_df.loc[0, "question"]

# Run retrieval for one sample question.
# best_document = rank 1 document.
# top_k_results = top 3 retrieved documents.
best_document, top_k_results = vector_retrieve(
    question=sample_question,
    document_store=document_store,
    document_embeddings=document_embeddings,
    top_k=3
)

print("Question:", sample_question)
print("Best retrieved document:", best_document["file_name"])
print("Top-k retrieved documents:")

for result in top_k_results:
    print(
        "Rank:", result["rank"],
        "| File:", result["file_name"],
        "| Score:", result["similarity_score"]
    )

Question: What are the default size limits for file uploads and total request size for the new multipart upload support on the OpenAI-compatible API endpoints?
Best retrieved document: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt
Top-k retrieved documents:
Rank: 1 | File: RAG_TC_001_dsid_ae068ee4aa9640159427cd941bef0238.txt | Score: 0.7318
Rank: 2 | File: RAG_TC_002_dsid_9550250a59e74f1bbd5612480b2e7100.txt | Score: 0.3154
Rank: 3 | File: RAG_TC_004_dsid_6c4c1c875e704f09b4d791d64d7bc7e5.txt | Score: 0.2331


In [27]:
# Step 31: Generate clean demo RAG response from retrieved document

# This step creates a clean actual_output from the retrieved document.
#
# Existing flow is kept:
# question -> retrieve best document -> generate clean response from that document
#
# Output format:
# Answer: <short direct answer>
# Source: <retrieved document id>

!pip install transformers torch -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the retrieved document content.
    """
    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


def clean_document_text_for_answer_generation(text):
    """
    Creates a cleaned temporary copy of retrieved document text.
    Original document content is not changed.
    """
    lines = str(text).splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if line.startswith("doc_id:"):
            continue

        if line.startswith("title:"):
            continue

        if line == "":
            continue

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)


# Load local instruction/text generation model.
# This model generates a clean answer from question + retrieved document.
#rag_response_model_name = "google/flan-t5-base"
rag_response_model_name = "google/flan-t5-large"

rag_tokenizer = AutoTokenizer.from_pretrained(rag_response_model_name)
rag_response_model = AutoModelForSeq2SeqLM.from_pretrained(rag_response_model_name)



def extract_relevant_answer(question, retrieved_contexts, top_n=3):
    """
    Generates a clean demo RAG response from the retrieved document.

    Input:
    - question: user question
    - document_text: retrieved document content
    - top_n: kept only for Step 32 compatibility

    Output:
    - clean answer with source document id
    """

    cleaned_contexts = [
        clean_document_text_for_answer_generation(context)
        for context in retrieved_contexts
    ]

    context = "\n\n".join(cleaned_contexts)[:2500]

    prompt = f"""
    You are a RAG assistant.

    Use the context to answer the question.
    Write the answer in 1 to 3 complete sentences.
    Include all important numbers, names, steps, or conditions needed to answer the question.
    Do not answer with only one word or a short phrase.
    Do not include unrelated context.

    Question:
    {question}

    Context:
    {context}

    Final Answer:
    """

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    with torch.no_grad():
        outputs = rag_response_model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=4,
            early_stopping=True
        )

    answer = rag_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    if answer == "":
        answer = "Answer not found in retrieved document."

    return f"Answer: {answer}"


# Test clean RAG response for first question using top-k documents.
best_document, top_k_results = vector_retrieve(
    question=sample_question,
    document_store=document_store,
    document_embeddings=document_embeddings,
    top_k=3
)

sample_top_k_contexts = [
    document["content"]
    for document in top_k_results
]

demo_answer = extract_relevant_answer(
    sample_question,
    sample_top_k_contexts,
    top_n=3
)

print("Generated demo answer:")
print(demo_answer)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generated demo answer:
Answer: 10MiB, 50MiB, 20.


In [28]:
print("qa_df exists:", "qa_df" in globals())

if "qa_df" in globals():
    print(qa_df.shape)
    print(qa_df.columns.tolist())

qa_df exists: True
(10, 21)
['test_id', 'input_code', 'question_id', 'question_type', 'question', 'expected_doc_ids', 'expected_doc_title', 'expected_source_type', 'expected_doc_content', 'gold_answer', 'answer_facts', 'actual_retrieved_doc_ids', 'actual_output', 'expected_embedding_code', 'actual_embedding_code', 'similarity_score', 'faithfulness_check', 'hallucination_check', 'match_status', 'remarks', 'expected_doc_path']


In [29]:
import pandas as pd

qa_df = pd.read_csv("lightweight_rag_qa_dataset_with_expected_codes.csv")

qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")

print("qa_df loaded:", qa_df.shape)

qa_df loaded: (10, 20)


In [30]:
# Step 32: Generate actual output using top-k retrieved documents

# This step runs the demo RAG pipeline for every question.
#
# Final flow:
# 1. Loop through every question in qa_df.
# 2. Use vector_retrieve() to retrieve top-k relevant documents.
# 3. Store top-k document IDs in actual_retrieved_doc_ids_ranked.
# 4. Keep top-k document contents in memory for LLM input and evaluation.
# 5. Send original question + top-k contents + prompt instructions to the LLM through extract_relevant_answer().
# 6. Store the LLM response in actual_output.


def extract_doc_id_from_file_content(content):
    """
    Extracts doc_id from the first line of exported document file.

    Example:
    doc_id: dsid_xxxxx

    Output:
    dsid_xxxxx
    """

    first_line = str(content).split("\n")[0]

    if first_line.startswith("doc_id:"):
        return first_line.replace("doc_id:", "").strip()

    return ""


# Create columns for top-k retrieval traceability.
qa_df["actual_retrieved_doc_ids_ranked"] = ""


# Store top-k retrieved contexts in memory for later evaluation.
retrieval_contexts_by_test_id = {}
retrieval_doc_ids_by_test_id = {}

for index, row in qa_df.iterrows():
    question = row["question"]

    # Retrieve top-k relevant documents for the question.
    best_document, top_k_results = vector_retrieve(
        question=question,
        document_store=document_store,
        document_embeddings=document_embeddings,
        top_k=3
    )

    # Extract top-k document IDs in ranked order.
    top_k_doc_ids = [
        extract_doc_id_from_file_content(document["content"])
        for document in top_k_results
    ]

    # Extract top-k document contents in ranked order.
    top_k_contexts = [
        document["content"]
        for document in top_k_results
    ]

    # Keep top-k contexts and document IDs in memory for later evaluation.
    retrieval_contexts_by_test_id[row["test_id"]] = top_k_contexts
    retrieval_doc_ids_by_test_id[row["test_id"]] = top_k_doc_ids

    # Send original question + top-k contents + prompt instructions to the LLM.
    # The prompt instructions are already inside extract_relevant_answer().
    generated_answer = extract_relevant_answer(
        question,
        top_k_contexts,
        top_n=3
    )

    # Store top-k retrieved document IDs.
    qa_df.loc[index, "actual_retrieved_doc_ids_ranked"] = str(top_k_doc_ids)


    # Store generated LLM response.
    qa_df.loc[index, "actual_output"] = generated_answer


# Display updated output.
display(
    qa_df[
        [
            "test_id",
            "question",
            "expected_doc_ids",
            "actual_retrieved_doc_ids_ranked",
            "actual_output"
        ]
    ]
)

,test_id,question,expected_doc_ids,actual_retrieved_doc_ids_ranked,actual_output
0,RAG_TC_001,What are the default size limits for file uplo...,dsid_ae068ee4aa9640159427cd941bef0238,"['dsid_ae068ee4aa9640159427cd941bef0238', 'dsi...","Answer: 10MiB, 50MiB, 20."
1,RAG_TC_002,What is the name of the new metric added so SR...,dsid_9550250a59e74f1bbd5612480b2e7100,"['dsid_9550250a59e74f1bbd5612480b2e7100', 'dsi...",Answer: Timeboxed.
2,RAG_TC_003,What are the acceptance criteria for the proje...,dsid_3fd6af404fae48e6b8ea5a57875ef78f,"['dsid_3fd6af404fae48e6b8ea5a57875ef78f', 'dsi...",Answer: Implementation should use runtime CSS ...
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,"['dsid_6c4c1c875e704f09b4d791d64d7bc7e5', 'dsi...",Answer: GCP emphasized keeping dimension names...
4,RAG_TC_005,What failover sequence and recovery targets di...,dsid_8e838ab6a98f4cbcb672d41f210ff89c,"['dsid_184be937d34a412ab5e61366d54d8ed6', 'dsi...","Answer: No-op, failover, partial shift, rollba..."
5,RAG_TC_006,In the draft spec about extending a routing po...,dsid_184be937d34a412ab5e61366d54d8ed6,"['dsid_184be937d34a412ab5e61366d54d8ed6', 'dsi...",Answer: Reachability: ok|degraded|down + confi...
6,RAG_TC_007,In a rolling investigation of a model regressi...,dsid_72ec4a9962ba43e88acd61abbba1052d,"['dsid_72ec4a9962ba43e88acd61abbba1052d', 'dsi...",Answer: The average triage rubric score change...
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,"['dsid_5fc2dba9f6ac4af2b49b4f546a4298d0', 'dsi...",Answer: 10m.
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",dsid_85deb10a652742baaf28af6149600001,"['dsid_85deb10a652742baaf28af6149600001', 'dsi...",Answer: Prepay bucket + seat licenses with an ...
9,RAG_TC_010,How does the new alerting approach group model...,dsid_c1a6a71323c04c1ba5445aadea340362,"['dsid_c1a6a71323c04c1ba5445aadea340362', 'dsi...",Answer: Introduces an alert route-matrix gener...


In [31]:
# Step 33: Save dataset with demo RAG generated outputs

# This CSV now contains:
# - expected answers
# - expected document IDs
# - actual retrieved document IDs from demo RAG
# - actual outputs from demo RAG
# - actual embedding codes

demo_rag_output_file = "rag_qa_dataset_with_demo_rag_outputs.csv"

qa_df.to_csv(demo_rag_output_file, index=False)

print("Demo RAG output CSV saved successfully:", demo_rag_output_file)

Demo RAG output CSV saved successfully: rag_qa_dataset_with_demo_rag_outputs.csv


## 8. Retrieval Document Check
This section checks whether the RAG system retrieved the expected source document.

In [32]:
# Step 28: Retrieval document check

# This logic checks whether the RAG system retrieved the correct source document.
# It works for all rows in the dataset.
# If actual_retrieved_doc_ids is empty, the row is marked as Not Run.

# Create result columns for retrieval evaluation.
qa_df["retrieval_match_status"] = ""
qa_df["retrieval_remarks"] = ""


def normalize_doc_ids(value):
    # Convert document ID values into a clean list.
    # This handles:
    # - empty values
    # - single document ID as text
    # - list-like document IDs saved as text

    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    # Remove brackets and quotes if IDs are stored like ['doc_id']
    value = value.replace("[", "").replace("]", "").replace("'", "").replace('"', "")

    # Split by comma in case multiple document IDs are present.
    doc_ids = [item.strip() for item in value.split(",") if item.strip()]

    return doc_ids


# Run retrieval check for every row.
for row_index, row in qa_df.iterrows():

    expected_doc_ids = normalize_doc_ids(row["expected_doc_ids"])
    actual_doc_ids = normalize_doc_ids(row["actual_retrieved_doc_ids"])

    # If actual retrieved document is not available, we cannot run retrieval check yet.
    if len(actual_doc_ids) == 0:
        qa_df.loc[row_index, "retrieval_match_status"] = "Not Run"
        qa_df.loc[row_index, "retrieval_remarks"] = "Actual retrieved document ID is not available yet."
        continue

    # Check whether any expected document ID is present in actual retrieved document IDs.
    is_match = any(doc_id in actual_doc_ids for doc_id in expected_doc_ids)

    if is_match:
        qa_df.loc[row_index, "retrieval_match_status"] = "Pass"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was retrieved."
    else:
        qa_df.loc[row_index, "retrieval_match_status"] = "Fail"
        qa_df.loc[row_index, "retrieval_remarks"] = "Expected document was not retrieved."


# Show retrieval check result.
display(
    qa_df[
        [
            "test_id",
            "expected_doc_ids",
            "actual_retrieved_doc_ids",
            "retrieval_match_status",
            "retrieval_remarks"
        ]
    ]
)

,test_id,expected_doc_ids,actual_retrieved_doc_ids,retrieval_match_status,retrieval_remarks
0,RAG_TC_001,dsid_ae068ee4aa9640159427cd941bef0238,NaN,Not Run,Actual retrieved document ID is not available ...
1,RAG_TC_002,dsid_9550250a59e74f1bbd5612480b2e7100,NaN,Not Run,Actual retrieved document ID is not available ...
2,RAG_TC_003,dsid_3fd6af404fae48e6b8ea5a57875ef78f,NaN,Not Run,Actual retrieved document ID is not available ...
3,RAG_TC_004,dsid_6c4c1c875e704f09b4d791d64d7bc7e5,NaN,Not Run,Actual retrieved document ID is not available ...
4,RAG_TC_005,dsid_8e838ab6a98f4cbcb672d41f210ff89c,NaN,Not Run,Actual retrieved document ID is not available ...
5,RAG_TC_006,dsid_184be937d34a412ab5e61366d54d8ed6,NaN,Not Run,Actual retrieved document ID is not available ...
6,RAG_TC_007,dsid_72ec4a9962ba43e88acd61abbba1052d,NaN,Not Run,Actual retrieved document ID is not available ...
7,RAG_TC_008,dsid_5fc2dba9f6ac4af2b49b4f546a4298d0,NaN,Not Run,Actual retrieved document ID is not available ...
8,RAG_TC_009,dsid_85deb10a652742baaf28af6149600001,NaN,Not Run,Actual retrieved document ID is not available ...
9,RAG_TC_010,dsid_c1a6a71323c04c1ba5445aadea340362,NaN,Not Run,Actual retrieved document ID is not available ...


In [33]:
""""

## 9. Context Precision Check

# This section checks whether the retrieved document/context is useful
# for answering the question.
#
# Simple meaning:
# - Retrieval check only checks whether the expected document ID was retrieved.
# - Context Precision checks whether the retrieved document content is useful.
#
# Correct comparison:
# gold_answer  vs  retrieved document content


def get_retrieved_document_content(retrieved_doc_id, document_store):
    """
    Finds the retrieved document content using actual_retrieved_doc_ids.

    Here document_store is a list of documents loaded from the document_store folder.
    """

    if pd.isna(retrieved_doc_id) or str(retrieved_doc_id).strip() == "":
        return ""

    retrieved_doc_id = str(retrieved_doc_id).strip()

    # Search each loaded document and check whether the doc_id is present inside it.
    for document in document_store:
        content = str(document["content"])

        if f"doc_id: {retrieved_doc_id}" in content:
            return content

    return ""

def context_precision_check(reference_answer, retrieved_context, threshold=0.60):
    """
    Compares expected answer meaning with retrieved document content.

    Input:
    - reference_answer: gold_answer / expected answer
    - retrieved_context: retrieved document content

    Output:
    - context precision score
    - Pass/Fail status
    - remarks
    """

    if pd.isna(reference_answer) or str(reference_answer).strip() == "":
        return "", "Not Run", "Reference answer is missing."

    if pd.isna(retrieved_context) or str(retrieved_context).strip() == "":
        return "", "Not Run", "Retrieved document content is missing."

    # Convert expected answer into embedding.
    reference_embedding = embedding_model.encode(
        str(reference_answer),
        convert_to_tensor=True
    )

    # Convert retrieved document content into embedding.
    context_embedding = embedding_model.encode(
        str(retrieved_context),
        convert_to_tensor=True
    )

    # Compare meaning similarity between expected answer and retrieved document content.
    score = float(util.cos_sim(reference_embedding, context_embedding)[0][0])
    score = round(score, 4)

    if score >= threshold:
        return score, "Pass", "Retrieved document content is useful for answering the question."

    return score, "Fail", "Retrieved document content is not useful enough for answering the question."


# Create columns for storing retrieved document content and Context Precision result.
qa_df["actual_retrieved_doc_content"] = ""
qa_df["context_precision_score"] = ""
qa_df["context_precision_status"] = ""
qa_df["context_precision_remarks"] = ""


# Run Context Precision check for every row.
for row_index, row in qa_df.iterrows():

    # Step 1: Get the retrieved document content using actual_retrieved_doc_ids.

    retrieved_context = get_retrieved_document_content(
        retrieved_doc_id=row["actual_retrieved_doc_ids"],
        document_store=document_store
    )

    # Step 2: Store retrieved document content for QA traceability.
    qa_df.loc[row_index, "actual_retrieved_doc_content"] = retrieved_context

    # Step 3: Compare gold_answer with retrieved document content.
    score, status, remarks = context_precision_check(
        reference_answer=row["gold_answer"],
        retrieved_context=retrieved_context
    )

    # Step 4: Store Context Precision result.
    qa_df.loc[row_index, "context_precision_score"] = score
    qa_df.loc[row_index, "context_precision_status"] = status
    qa_df.loc[row_index, "context_precision_remarks"] = remarks


# Show Context Precision result.
display(
    qa_df[
        [
            "test_id",
            "gold_answer",
            "actual_retrieved_doc_ids",
            "context_precision_score",
            "context_precision_status",
            "context_precision_remarks"
        ]
    ]
)

""""

IndentationError: unexpected indent (2654566420.py, line 18)

In [37]:
# Gemini model setup for DeepEval
# This lets DeepEval use Gemini instead of OpenAI.

!pip install google-genai -q

from google import genai
from google.colab import userdata
from deepeval.models import DeepEvalBaseLLM


class GeminiDeepEvalModel(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-2.0-flash"):
        self.model_name = model_name
        self.client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )
        return response.text

    async def a_generate(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )
        return response.text

    def get_model_name(self):
        return self.model_name


gemini_model = GeminiDeepEvalModel()

print("Gemini model connected for DeepEval.")

Gemini model connected for DeepEval.


In [38]:
## 9. Contextual Precision Check using DeepEval

# This section uses DeepEval's professional RAG metric.
#
# Contextual Precision checks whether the useful retrieved contexts
# are ranked higher than less useful/noisy contexts.
#
# It compares:
# - input              -> question
# - expected_output    -> gold_answer
# - retrieval_context  -> top-k retrieved document contents
#
# Note:
# actual_output is required by DeepEval's LLMTestCase structure,
# but Contextual Precision mainly evaluates retrieval_context ranking.

!pip install deepeval -q

from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric


# Create result columns.
qa_df["contextual_precision_score"] = ""
qa_df["contextual_precision_status"] = ""
qa_df["contextual_precision_reason"] = ""


# Create DeepEval Contextual Precision metric.
contextual_precision_metric = ContextualPrecisionMetric(
    threshold=0.7,
    model=gemini_model,
    include_reason=True
)


for row_index, row in qa_df.iterrows():

    test_id = row["test_id"]

    retrieval_context = retrieval_contexts_by_test_id.get(test_id, [])

    if len(retrieval_context) == 0:
        qa_df.loc[row_index, "contextual_precision_status"] = "Not Run"
        qa_df.loc[row_index, "contextual_precision_reason"] = "Top-k retrieval context is missing."
        continue

    test_case = LLMTestCase(
        input=row["question"],
        actual_output=row["actual_output"],
        expected_output=row["gold_answer"],
        retrieval_context=retrieval_context
    )

    contextual_precision_metric.measure(test_case)

    qa_df.loc[row_index, "contextual_precision_score"] = contextual_precision_metric.score
    qa_df.loc[row_index, "contextual_precision_reason"] = contextual_precision_metric.reason

    if contextual_precision_metric.score >= 0.7:
        qa_df.loc[row_index, "contextual_precision_status"] = "Pass"
    else:
        qa_df.loc[row_index, "contextual_precision_status"] = "Fail"


display(
    qa_df[
        [
            "test_id",
            "question",
            "gold_answer",
            "actual_retrieved_doc_ids_ranked",
            "contextual_precision_score",
            "contextual_precision_status",
            "contextual_precision_reason"
        ]
    ]
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.3.3 which is incompatible.


Output()

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 52.76269689s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goo

In [35]:
## 10. Response Relevancy Check

# This section checks whether actual_output is relevant to the original question.
#
# Ragas-style logic:
# 1. Take actual_output.
# 2. Use a local Hugging Face question-generation model to generate questions.
# 3. Convert generated questions and original question into embeddings.
# 4. Compare them using cosine similarity.
# 5. Average the score and mark Pass/Fail.
#
# This version does not use OpenAI API or Gemini API.
# It runs locally inside Colab after downloading the Hugging Face model.

!pip install transformers sentencepiece -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np


# Load local Hugging Face question-generation model.
# This model is used to generate questions from actual_output.
question_generation_model_name = "valhalla/t5-small-qg-hl"

qg_tokenizer = AutoTokenizer.from_pretrained(question_generation_model_name)
qg_model = AutoModelForSeq2SeqLM.from_pretrained(question_generation_model_name)


def generate_questions_from_answer(actual_output, question_count=3):
    """
    Generates possible questions from actual_output using a local Hugging Face model.

    Input:
    - actual_output: answer generated by the demo RAG pipeline
    - question_count: number of questions to generate

    Output:
    - list of generated questions
    """

    if pd.isna(actual_output) or str(actual_output).strip() == "":
        return []

    # Keep the text short enough for the model.
    #answer_text = str(actual_output).strip()[:1000]
    answer_text = str(actual_output).strip()

    # If actual_output has Answer and Source, use only Answer part.
    if "Source:" in answer_text:
        answer_text = answer_text.split("Source:")[0].strip()

    # Remove Answer label before sending to question generator.
    answer_text = answer_text.replace("Answer:", "").strip()

    # Keep text short enough for the question generation model.
    answer_text = answer_text[:1000]

    # Prompt format for question generation.
    input_text = "generate question: " + answer_text

    inputs = qg_tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    outputs = qg_model.generate(
        **inputs,
        max_length=64,
        num_return_sequences=question_count,
        num_beams=5,
        do_sample=True
    )

    generated_questions = [
        qg_tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

    return generated_questions


def response_relevancy_check(original_question, actual_output, threshold=0.60):
    """
    Checks whether actual_output is relevant to the original question.

    Input:
    - original_question: original question from dataset
    - actual_output: answer generated by demo RAG pipeline

    Output:
    - generated questions
    - response relevancy score
    - Pass/Fail status
    - remarks
    """

    if pd.isna(original_question) or str(original_question).strip() == "":
        return [], "", "Not Run", "Original question is missing."

    if pd.isna(actual_output) or str(actual_output).strip() == "":
        return [], "", "Not Run", "Actual output is missing."

    # Step 1: Generate questions from actual_output.
    generated_questions = generate_questions_from_answer(actual_output)

    if len(generated_questions) == 0:
        return [], "", "Not Run", "Question generation failed."

    # Step 2: Convert original question into embedding.
    original_question_embedding = embedding_model.encode(
        str(original_question),
        convert_to_tensor=True
    )

    similarity_scores = []

    # Step 3: Compare each generated question with original question.
    for generated_question in generated_questions:

        generated_question_embedding = embedding_model.encode(
            str(generated_question),
            convert_to_tensor=True
        )

        score = float(
            util.cos_sim(
                original_question_embedding,
                generated_question_embedding
            )[0][0]
        )

        similarity_scores.append(score)

    # Step 4: Average all similarity scores.
    final_score = round(float(np.mean(similarity_scores)), 4)

    if final_score >= threshold:
        return generated_questions, final_score, "Pass", "Actual output is relevant to the question."

    return generated_questions, final_score, "Fail", "Actual output is not relevant enough to the question."


# Create columns for Response Relevancy result.
qa_df["generated_question_from_answer"] = ""
qa_df["response_relevancy_score"] = ""
qa_df["response_relevancy_status"] = ""
qa_df["response_relevancy_remarks"] = ""


# Run Response Relevancy check for every row.
for row_index, row in qa_df.iterrows():

    generated_questions, score, status, remarks = response_relevancy_check(
        original_question=row["question"],
        actual_output=row["actual_output"]
    )

    qa_df.loc[row_index, "generated_question_from_answer"] = str(generated_questions)
    qa_df.loc[row_index, "response_relevancy_score"] = score
    qa_df.loc[row_index, "response_relevancy_status"] = status
    qa_df.loc[row_index, "response_relevancy_remarks"] = remarks


# Show Response Relevancy result.
display(
    qa_df[
        [
            "test_id",
            "question",
            "actual_output",
            "generated_question_from_answer",
            "response_relevancy_score",
            "response_relevancy_status",
            "response_relevancy_remarks"
        ]
    ]
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 4.1.4 requires click<8.4.0,>=8.0.0, but you have click 8.4.2 which is incompatible.


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

,test_id,question,actual_output,generated_question_from_answer,response_relevancy_score,response_relevancy_status,response_relevancy_remarks
0,RAG_TC_001,What are the default size limits for file uplo...,"Answer: 10MiB, 50MiB, 20.","['What does 10MiB, 50MiB, 20MiB, 50MiB, 20MiB,...",0.1321,Fail,Actual output is not relevant enough to the qu...
1,RAG_TC_002,What is the name of the new metric added so SR...,Answer: Timeboxed.,"['What is Timeboxed?', 'What is the Timeboxed?...",0.2875,Fail,Actual output is not relevant enough to the qu...
2,RAG_TC_003,What are the acceptance criteria for the proje...,Answer: Implementation should use runtime CSS ...,['What should implementation use runtime CSS v...,0.1942,Fail,Actual output is not relevant enough to the qu...
3,RAG_TC_004,In the meeting about onboarding a SaaS product...,Answer: GCP emphasized keeping dimension names...,['What did GCP emphasize keeping dimension nam...,0.2628,Fail,Actual output is not relevant enough to the qu...
4,RAG_TC_005,What failover sequence and recovery targets di...,"Answer: No-op, failover, partial shift, rollba...",['What is the executable change emitted by con...,-0.0775,Fail,Actual output is not relevant enough to the qu...
5,RAG_TC_006,In the draft spec about extending a routing po...,Answer: Reachability: ok|degraded|down + confi...,['What is the reachability of ok|degraded|down...,0.3547,Fail,Actual output is not relevant enough to the qu...
6,RAG_TC_007,In a rolling investigation of a model regressi...,Answer: The average triage rubric score change...,['What was the average triage rubric score cha...,0.6868,Pass,Actual output is relevant to the question.
7,RAG_TC_008,"In the internal shiproom runner notes, what is...",Answer: 10m.,"['What does 10m mean?', 'What is the theme of ...",0.1029,Fail,Actual output is not relevant enough to the qu...
8,RAG_TC_009,"In the EdgePath evaluation email thread, what ...",Answer: Prepay bucket + seat licenses with an ...,['What does the Prepay bucket + seat license h...,0.2548,Fail,Actual output is not relevant enough to the qu...
9,RAG_TC_010,How does the new alerting approach group model...,Answer: Introduces an alert route-matrix gener...,['What is the name of the alert route-matrix g...,0.513,Fail,Actual output is not relevant enough to the qu...


## 9. Semantic Similarity Comparison
This section compares expected answers and actual outputs using embedding similarity score.

In [ ]:
# Step 28: Compare expected answers and actual outputs using embeddings

# This comparison logic is built for the full dataset.
# It will run only for rows where actual_output is available.
# Rows without actual_output will remain as "Not Run".

import numpy as np


# Make sure these columns can store text/results properly.
qa_df["actual_output"] = qa_df["actual_output"].astype("object")
qa_df["actual_embedding_code"] = qa_df["actual_embedding_code"].astype("object")
qa_df["similarity_score"] = qa_df["similarity_score"].astype("object")
qa_df["match_status"] = qa_df["match_status"].astype("object")
qa_df["remarks"] = qa_df["remarks"].astype("object")


# Similarity threshold.
# If score is 0.80 or above, we mark it as Pass.
# You can tune this later based on project requirement.
SIMILARITY_THRESHOLD = 0.80


def calculate_similarity(expected_text, actual_text):
    # Convert expected answer into embedding vector.
    expected_vector = embedding_model.encode(str(expected_text))

    # Convert actual answer into embedding vector.
    actual_vector = embedding_model.encode(str(actual_text))

    # Calculate cosine similarity.
    # Higher score means both answers have closer meaning.
    similarity = np.dot(expected_vector, actual_vector) / (
        np.linalg.norm(expected_vector) * np.linalg.norm(actual_vector)
    )

    return round(float(similarity), 4)


# Loop through every test case in the QA dataset.
for row_index, row in qa_df.iterrows():

    # If actual_output is empty, we cannot compare this row yet.
    if pd.isna(row["actual_output"]) or str(row["actual_output"]).strip() == "":
        qa_df.loc[row_index, "match_status"] = "Not Run"
        qa_df.loc[row_index, "remarks"] = "Actual output is not available yet."
        continue

    # Create actual embedding fingerprint code from actual_output.
    qa_df.loc[row_index, "actual_embedding_code"] = create_embedding_fingerprint(
        row["actual_output"],
        prefix="ACT"
    )

    # Calculate similarity between gold_answer and actual_output.
    score = calculate_similarity(
        row["gold_answer"],
        row["actual_output"]
    )

    qa_df.loc[row_index, "similarity_score"] = score

    # Mark Pass/Fail based on similarity threshold.
    if score >= SIMILARITY_THRESHOLD:
        qa_df.loc[row_index, "match_status"] = "Pass"
        qa_df.loc[row_index, "remarks"] = "Actual output is semantically similar to expected answer."
    else:
        qa_df.loc[row_index, "match_status"] = "Fail"
        qa_df.loc[row_index, "remarks"] = "Actual output is not similar enough to expected answer."


# Show final comparison status.
display(
    qa_df[
        [
            "test_id",
            "gold_answer",
            "actual_output",
            "expected_embedding_code",
            "actual_embedding_code",
            "similarity_score",
            "match_status",
            "remarks"
        ]
    ]
)

In [ ]:
# Step 29: Save final demo comparison result

# This CSV includes:
# - expected answers
# - expected embedding codes
# - one demo actual output
# - actual embedding code
# - similarity score
# - pass/fail result for the demo row

qa_df.to_csv("rag_qa_demo_comparison_result.csv", index=False)

print("Final demo comparison CSV saved: rag_qa_demo_comparison_result.csv")

In [ ]:
# Download the final CSV from Colab to your computer

from google.colab import files

files.download("rag_qa_demo_comparison_result.csv")

 Fact-Level Meaning Check

This section checks whether the expected answer facts are present in the actual output using sentence-level embedding similarity.

## 10. Fact-Level Meaning Check
This section checks whether expected answer facts are present in the actual output using sentence-level embedding similarity.

## Overall QA Status
This section combines retrieval check, semantic similarity check, and fact-level check into one final QA status.

In [ ]:
# Overall QA Status

# This logic combines three checks:
# 1. retrieval_match_status -> whether correct document was retrieved
# 2. match_status -> whether actual answer is semantically similar to expected answer
# 3. fact_check_status -> whether expected facts are present in actual answer

qa_df["overall_qa_status"] = ""
qa_df["overall_qa_remarks"] = ""


for row_index, row in qa_df.iterrows():

    retrieval_status = str(row.get("retrieval_match_status", "")).strip()
    similarity_status = str(row.get("match_status", "")).strip()
    fact_status = str(row.get("fact_check_status", "")).strip()

    failed_checks = []
    not_run_checks = []

    # Track checks that are not completed
    if retrieval_status == "" or retrieval_status == "Not Run":
        not_run_checks.append("retrieval check")

    if similarity_status == "" or similarity_status == "Not Run":
        not_run_checks.append("semantic similarity check")

    if fact_status == "" or fact_status == "Not Run":
        not_run_checks.append("fact-level check")

    # If any required check is not run, overall status is Not Run.
    if len(not_run_checks) > 0:
        qa_df.loc[row_index, "overall_qa_status"] = "Not Run"
        qa_df.loc[row_index, "overall_qa_remarks"] = (
            "Not completed: " + ", ".join(not_run_checks)
        )
        continue

    # Track failed checks
    if retrieval_status == "Fail":
        failed_checks.append("retrieval check failed")

    if similarity_status == "Fail":
        failed_checks.append("semantic similarity check failed")

    if fact_status == "Fail":
        failed_checks.append("fact-level check failed")

    # If all checks pass, overall status is Pass.
    if (
        retrieval_status == "Pass"
        and similarity_status == "Pass"
        and fact_status == "Pass"
    ):
        qa_df.loc[row_index, "overall_qa_status"] = "Pass"
        qa_df.loc[row_index, "overall_qa_remarks"] = (
            "Retrieval, semantic similarity, and fact-level checks passed."
        )
        continue

    # If fact check is partial, overall status is Partial.
    if fact_status == "Partial":
        qa_df.loc[row_index, "overall_qa_status"] = "Partial"

        if len(failed_checks) > 0:
            qa_df.loc[row_index, "overall_qa_remarks"] = (
                "; ".join(failed_checks) + "; fact-level check partially passed"
            )
        else:
            qa_df.loc[row_index, "overall_qa_remarks"] = (
                "Fact-level check partially passed; some expected facts are missing."
            )
        continue

    # If any check fails, overall status is Fail.
    if len(failed_checks) > 0:
        qa_df.loc[row_index, "overall_qa_status"] = "Fail"
        qa_df.loc[row_index, "overall_qa_remarks"] = "; ".join(failed_checks)
        continue


# Show final QA status.
display(
    qa_df[
        [
            "test_id",
            "retrieval_match_status",
            "match_status",
            "fact_check_status",
            "overall_qa_status",
            "overall_qa_remarks"
        ]
    ]
)

In [ ]:
# Save updated result CSV with similarity + fact-level check results

qa_df.to_csv("rag_qa_final_demo_result_with_all_checks.csv", index=False)

print("Saved: rag_qa_final_demo_result_with_all_checks.csv")

In [ ]:
import os

print(os.path.exists("rag_qa_final_demo_result_with_all_checks.csv"))
print(os.path.getsize("rag_qa_final_demo_result_with_all_checks.csv"))
# Download updated CSV to your computer

from google.colab import files

files.download("rag_qa_final_demo_result_with_all_checks.csv")

## DeepEval Task Completion Learning Experiment

This section is only a learning experiment. It checks whether DeepEval Task Completion can be applied to our RAG flow by treating one RAG run as a simple task flow. This is not part of the main RAG evaluation result.


In [ ]:
# DeepEval Task Completion Learning Experiment

# This is only for learning.
# It runs on one row only and does not change the main QA result.

import os

try:
    from deepeval.tracing import observe
    from deepeval.dataset import Golden, EvaluationDataset
    from deepeval.metrics import TaskCompletionMetric

    # DeepEval Task Completion uses an LLM judge.
    # Keep model explicit for cost/control.
    task_completion_metric = TaskCompletionMetric(
        threshold=0.7,
        model="gpt-4o-mini",
        task="Answer the user question correctly using the retrieved document."
    )

    # Use only the first test case for this learning experiment.
    sample_row = qa_df.iloc[0]

    @observe(type="agent")
    def rag_task_completion_experiment(user_question):
        # Treat existing RAG output as the final outcome of one task.
        return sample_row["actual_output"]

    experiment_dataset = EvaluationDataset(
        goldens=[
            Golden(input=sample_row["question"])
        ]
    )

    print("Running Task Completion learning experiment for:")
    print(sample_row["test_id"])

    for golden in experiment_dataset.evals_iterator(metrics=[task_completion_metric]):
        rag_task_completion_experiment(golden.input)

    print("Task Completion Score:", task_completion_metric.score)
    print("Task Completion Reason:", task_completion_metric.reason)

except Exception as error:
    print("Task Completion experiment was not completed.")
    print("Reason:", error)
    print("Note: This metric needs DeepEval tracing and an LLM API key/model setup.")
